<a href="https://colab.research.google.com/github/karishma0710-A/Datascience_projects/blob/main/fake_and_real_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
print("Setup Complete")

In [ ]:
# Step 1: Upload ZIP file
from google.colab import files
uploaded = files.upload()  # Choose your dataset.zip file

# Step 2: Unzip the file
import zipfile
import io

for filename in uploaded.keys():
    with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
        zip_ref.extractall()  # Extract in current directory

# Step 3: Read the CSV files
import pandas as pd
df_fake = pd.read_csv("Fake.csv")
df_true = pd.read_csv("True.csv")

# Step 4: Add labels and combine
df_fake["label"] = "FAKE"
df_true["label"] = "REAL"
df = pd.concat([df_fake, df_true], axis=0)

print(df.head())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Split data into training & testing
x_train, x_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=7
)

# Convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train = vectorizer.fit_transform(x_train)
tfidf_test = vectorizer.transform(x_test)

# Train model
model = PassiveAggressiveClassifier(max_iter=50)
model.fit(tfidf_train, y_train)

# Predict & check accuracy
y_pred = model.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f"Accuracy: {score*100:.2f}%")

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
sample_text = ["Donald Trump meets Elon Musk in New York."]
sample_vector = vectorizer.transform(sample_text)
print("Prediction:", model.predict(sample_vector)[0])


In [ ]:
# Ask user for a news headline/article
user_input = input("Enter a news headline or article: ")

# Transform and predict
user_vector = vectorizer.transform([user_input])
prediction = model.predict(user_vector)[0]

print("Prediction:", prediction)


In [ ]:
!pip install gradio

import gradio as gr

def classify(text):
    return "FAKE" if "fake" in text.lower() else "REAL"

iface = gr.Interface(fn=classify, inputs="text", outputs="text")
iface.launch(share=True)

#AAAA(uppercase)- REAL
#aaaa(lowercase)- fake